# 04. Advanced Graph Algorithms: SCC, Bridges, Euler Path

Problems at the hard end of graph interviews.

**Learning order:** SCC (KOSARAJU) → BRIDGES (TARJAN) → EULER PATH → INTERVIEW THINKING

## 1. Strongly Connected Components (SCC)

In a **directed** graph, an SCC is a maximal set of vertices where **every vertex can reach every other**. Real life: groups of web pages that all link (indirectly) to each other.

**Kosaraju's algorithm** (two DFS passes):
1. DFS on the graph, record vertices by **finish time** (push to a stack when done).
2. Build the **reversed** graph. Pop vertices from the stack; each unvisited vertex's DFS on the reversed graph is one SCC.

O(V + E).

In [1]:
from collections import defaultdict

def kosaraju_scc(n, edges):
    graph, rgraph = defaultdict(list), defaultdict(list)
    for u, v in edges:
        graph[u].append(v)
        rgraph[v].append(u)
    visited, order = [False] * n, []
    def dfs1(u):
        visited[u] = True
        for v in graph[u]:
            if not visited[v]:
                dfs1(v)
        order.append(u)                                    # finish time order
    for u in range(n):
        if not visited[u]:
            dfs1(u)
    visited = [False] * n
    comps = []
    def dfs2(u, comp):
        visited[u] = True
        comp.append(u)
        for v in rgraph[u]:
            if not visited[v]:
                dfs2(v, comp)
    for u in reversed(order):
        if not visited[u]:
            comp = []
            dfs2(u, comp)
            comps.append(sorted(comp))
    return sorted(comps)

edges = [(0, 1), (1, 2), (2, 0), (1, 3), (3, 4), (4, 5), (5, 3)]
print(kosaraju_scc(6, edges))   # [[0, 1, 2], [3, 4, 5]]

[[0, 1, 2], [3, 4, 5]]


## 2. Bridges (Critical Connections)

A **bridge** is an edge whose removal **disconnects** the graph (a single point of failure in a network).

**Tarjan's low-link idea (one DFS):**
- `disc[u]` = time when `u` is first visited.
- `low[u]` = the earliest `disc` reachable from the subtree of `u` using **at most one back edge**.
- Edge `(u, v)` (where `v` is a DFS child of `u`) is a bridge if `low[v] > disc[u]`: the subtree of `v` cannot reach `u` or above without this edge.

Articulation points use the similar condition `low[v] >= disc[u]`.

In [2]:
def critical_connections(n, connections):
    graph = defaultdict(list)
    for u, v in connections:
        graph[u].append(v)
        graph[v].append(u)
    disc, low, timer, bridges = [-1] * n, [0] * n, [0], []
    def dfs(u, parent):
        disc[u] = low[u] = timer[0]
        timer[0] += 1
        for v in graph[u]:
            if v == parent:
                continue
            if disc[v] == -1:
                dfs(v, u)
                low[u] = min(low[u], low[v])
                if low[v] > disc[u]:
                    bridges.append([u, v])
            else:
                low[u] = min(low[u], disc[v])              # back edge
    dfs(0, -1)
    return bridges

print(critical_connections(4, [[0, 1], [1, 2], [2, 0], [1, 3]]))   # [[1, 3]]

[[1, 3]]


## 3. Eulerian path: Reconstruct Itinerary

Use **every edge exactly once**. Hierholzer's algorithm: DFS, always take the next unused edge (here the smallest destination alphabetically), and add a vertex to the answer **when it has no more outgoing edges** (post-order). Reverse at the end.

In [3]:
import heapq

def find_itinerary(tickets):
    graph = defaultdict(list)
    for src, dst in tickets:
        heapq.heappush(graph[src], dst)                    # smallest destination first
    route = []
    def dfs(airport):
        while graph[airport]:
            dfs(heapq.heappop(graph[airport]))
        route.append(airport)                              # post-order
    dfs("JFK")
    return route[::-1]

print(find_itinerary([["MUC", "LHR"], ["JFK", "MUC"], ["SFO", "SJC"], ["LHR", "SFO"]]))   # ['JFK', 'MUC', 'LHR', 'SFO', 'SJC']

['JFK', 'MUC', 'LHR', 'SFO', 'SJC']


## 4. Interview thinking

**Signals:** "groups that can reach each other" (SCC), "single point of failure" (bridge / articulation point), "use every edge/ticket once" (Euler path).

**Say:** "A DFS with discovery and low-link times finds bridges in O(V + E)."

**Common mistakes:** using the parent check wrongly with **multi-edges** (compare edge ids, not vertex ids), recursion limits on large graphs (iterative DFS or `sys.setrecursionlimit`), forgetting the reversed graph in Kosaraju.

These are asked mainly at senior/FAANG levels. Know the idea and be able to code bridges.

## Quick Revision

- SCC: Kosaraju = finish order + DFS on the reversed graph.
- Bridge: `low[v] > disc[u]`. Articulation: `low[v] >= disc[u]`.
- Euler path: Hierholzer, append on exit, reverse.